# CivicAI — ML vs Qwen2.5 vs Hybrid Evaluation

**Purpose:** Reproducibly assess complaint **category classification**, **priority prediction**, **department routing**, and **summary quality** in the CivicAI academic prototype.

**Important:** No measured accuracy is claimed in this notebook until you supply an independently labeled evaluation CSV and run the cells. The included sample CSV is **only a formatting example**, not an independent benchmark. Do not publish scores calculated from it.

**Models:** existing `ml_model.predict_category`, existing `local_llm_service.analyze_with_local_llm` (Ollama / Qwen2.5), and an explicit hybrid decision rule. The hybrid rule here is a **research baseline**, not necessarily the same decision logic used in `app.py`.

**Privacy:** Run on synthetic or consented, de-identified complaints. Local LLM calls go to the Ollama endpoint configured by `OLLAMA_URL`.

## 1. Setup

Save this notebook in `C:\Users\pavit\CivicAI_VisualMatch` alongside `app.py`, `ml_model.py`, and `local_llm_service.py`. In VS Code, select the project's `.venv` Python kernel. Run Ollama and ensure `qwen2.5:1.5b` is installed. If packages are missing, run the command below **in PowerShell**, not in a notebook cell:

```powershell
.\.venv\Scripts\python.exe -m pip install notebook ipykernel pandas numpy scikit-learn matplotlib seaborn
```

Keep the scikit-learn version compatible with the saved ML model (your model was trained using `scikit-learn==1.8.0`).

In [ ]:
from pathlib import Path
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

ROOT = Path.cwd()
if not (ROOT / 'ml_model.py').exists():
    raise FileNotFoundError('Open this notebook from the CivicAI_VisualMatch project folder.')
sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / 'evaluation_results'
OUTPUT.mkdir(exist_ok=True)
print('Project:', ROOT)
print('Python:', sys.executable)

## 2. Create a CSV template (first run only)

Edit `civicai_evaluation.csv` with **at least 100–200 independent, human-labeled complaints** (preferably more), balanced across all categories and priorities. **Do not use training examples or LLM-generated labels as ground truth.** The initial rows below illustrate the required format and must be replaced before reporting results.

Required columns: `complaint_id`, `description`, `true_category`, `true_priority`. Optional: `true_department`, `reference_summary`. The priority labels must be `Low`, `Medium`, or `High` and defined consistently by your annotators.

In [ ]:
DATA_FILE = ROOT / 'civicai_evaluation.csv'
if not DATA_FILE.exists():
    examples = [
        ('EXAMPLE-01', 'The streetlight outside our apartment has been off for four nights.', 'Street Lighting', 'Medium'),
        ('EXAMPLE-02', 'A large pothole on the main road is causing vehicles to swerve.', 'Roads & Potholes', 'High'),
        ('EXAMPLE-03', 'Garbage bins have not been collected for five days.', 'Garbage & Waste', 'Medium'),
        ('EXAMPLE-04', 'There is no water supply to our street since yesterday.', 'Water Supply', 'Medium'),
        ('EXAMPLE-05', 'Drainage water is overflowing across the footpath.', 'Drainage & Sewage', 'High'),
        ('EXAMPLE-06', 'A fallen tree branch is blocking the entrance to the park.', 'Parks & Trees', 'Medium'),
        ('EXAMPLE-07', 'An uncovered electrical wire is sparking near pedestrians.', 'Public Safety', 'High'),
    ]
    pd.DataFrame(examples, columns=['complaint_id', 'description', 'true_category', 'true_priority']).to_csv(DATA_FILE, index=False)
    print('Created EXAMPLE-ONLY template:', DATA_FILE)
else:
    print('Using existing file:', DATA_FILE)
print(pd.read_csv(DATA_FILE).head().to_string(index=False))

## 3. Load and validate independent ground truth

The notebook refuses to publish summary scores if example IDs remain. You may still inspect model predictions for debugging. Categories are imported from the project's actual `genai_service.py`.

In [ ]:
from genai_service import CATEGORIES, PRIORITIES, DEPARTMENTS
from ml_model import predict_category
from local_llm_service import analyze_with_local_llm

df = pd.read_csv(DATA_FILE, dtype=str).fillna('')
required = {'complaint_id','description','true_category','true_priority'}
missing = required - set(df.columns)
if missing: raise ValueError(f'Missing required columns: {sorted(missing)}')
if df.empty: raise ValueError('Evaluation CSV is empty.')
if df['complaint_id'].duplicated().any(): raise ValueError('Duplicate complaint_id values.')
if df['description'].str.strip().eq('').any(): raise ValueError('Blank complaint description found.')
invalid_cat = sorted(set(df.true_category) - set(CATEGORIES))
invalid_pri = sorted(set(df.true_priority) - set(PRIORITIES))
if invalid_cat: raise ValueError(f'Unknown true_category labels: {invalid_cat}; allowed: {CATEGORIES}')
if invalid_pri: raise ValueError(f'Unknown true_priority labels: {invalid_pri}; allowed: {PRIORITIES}')
if 'true_department' not in df: df['true_department'] = df.true_category.map(DEPARTMENTS)
if 'reference_summary' not in df: df['reference_summary'] = ''
IS_DEMO = df.complaint_id.str.startswith('EXAMPLE-').any()
print('Rows:',len(df),'| Contains demonstration examples:',IS_DEMO)
print('Category distribution:\n', df.true_category.value_counts().to_string())
print('Priority distribution:\n', df.true_priority.value_counts().to_string())
if IS_DEMO:
    print('WARNING: Template examples present. Results are diagnostic only, NOT research accuracy.')
elif len(df)<100:
    print('WARNING: Fewer than 100 independent cases. Treat results as exploratory.')

## 4. Run the three models

**ML:** predicts a category using your saved TF-IDF + LinearSVC pipeline.

**Qwen:** generates category, priority, department, and summary. Calls are sequential to avoid overwhelming local Ollama. You can limit a debugging run using `MAX_CASES = 5`; set it to `None` for the full evaluation.

**Hybrid baseline:** uses the ML category when ML and Qwen disagree; otherwise uses their agreed category. This is intentionally conservative and is **not a learned fusion model**. As written, hybrid category predictions will be identical to ML; the notebook checks this explicitly. To demonstrate a genuine improvement, you would need a defensible fusion rule validated on a separate development set (e.g., confidence-calibrated arbitration). Hybrid priority and summary come from Qwen.

Failures are recorded, not silently treated as correct.

In [ ]:
MAX_CASES = 5  # Change to None to evaluate the entire CSV
RETRY_FAILED_ONCE = True
subset = df.head(MAX_CASES).copy() if MAX_CASES is not None else df.copy()
rows=[]
for idx, row in enumerate(subset.itertuples(index=False), 1):
    item = row._asdict()
    complaint = item['description']
    result = dict(item)
    try:
        result['ml_category'] = predict_category(complaint)
        result['ml_error'] = ''
    except Exception as e:
        result['ml_category'] = ''
        result['ml_error'] = f'{type(e).__name__}: {e}'
    llm=None; error=''; start=time.perf_counter()
    for attempt in range(2 if RETRY_FAILED_ONCE else 1):
        try:
            llm = analyze_with_local_llm(complaint)
            break
        except Exception as e:
            error = f'{type(e).__name__}: {e}'
    result['llm_latency_seconds'] = round(time.perf_counter()-start,3)
    result['llm_error'] = '' if llm else error
    result['llm_category'] = llm.get('category','') if llm else ''
    result['llm_priority'] = llm.get('priority','') if llm else ''
    result['llm_department'] = llm.get('department','') if llm else ''
    result['llm_summary'] = llm.get('summary','') if llm else ''
    result['llm_citizen_reply'] = llm.get('citizen_reply','') if llm else ''
    # Conservative fusion baseline: ML wins disagreements.
    result['hybrid_category'] = result['ml_category'] if result['ml_category'] else result['llm_category']
    result['hybrid_priority'] = result['llm_priority']
    result['hybrid_department'] = DEPARTMENTS.get(result['hybrid_category'], '')
    result['models_agree'] = bool(result['ml_category'] and result['ml_category']==result['llm_category'])
    rows.append(result)
    print(f'{idx}/{len(subset)} {result["complaint_id"]}: ML={result["ml_category"] or "ERROR"}; '
          f'Qwen={result["llm_category"] or "ERROR"}; {result["llm_latency_seconds"]}s')
pred = pd.DataFrame(rows)
pred.to_csv(OUTPUT / 'predictions.csv', index=False)
print('Saved:', OUTPUT / 'predictions.csv')

## 5. Evaluation metrics and confusion matrices

For model comparisons, evaluate on the **same successfully processed rows**. Also report coverage, because excluding failed predictions can inflate accuracy. The classification report uses macro-F1 to account for class imbalance. **Priority accuracy applies only to Qwen and hybrid**, since the traditional ML model does not predict priority.

In [ ]:
def metrics(y_true, y_pred, labels):
    p,r,f,_ = precision_recall_fscore_support(y_true,y_pred,labels=labels,average='macro',zero_division=0)
    return {'accuracy':accuracy_score(y_true,y_pred), 'macro_precision':p,
            'macro_recall':r, 'macro_f1':f, 'n':len(y_true)}

print('Processed:',len(pred),'of',len(df),'total labeled cases')
for col in ['ml_category','llm_category','hybrid_category']:
    valid = pred[col].isin(CATEGORIES)
    print(f'{col} coverage: {valid.sum()}/{len(pred)} ({valid.mean():.1%})')

common = pred[pred[['ml_category','llm_category','hybrid_category']].isin(CATEGORIES).all(axis=1)].copy()
summary_rows=[]
if len(common):
    for model,col in [('Traditional ML','ml_category'),('Qwen2.5 LLM','llm_category'),('Hybrid (ML-first)','hybrid_category')]:
        summary_rows.append({'model':model,**metrics(common.true_category,common[col],CATEGORIES)})
    summary = pd.DataFrame(summary_rows)
    print('DIAGNOSTIC ONLY — NOT FOR PUBLICATION' if IS_DEMO else 'CATEGORY METRICS')
    display(summary)
    summary.to_csv(OUTPUT / 'category_metrics.csv',index=False)
    assert (common['hybrid_category']==common['ml_category']).all(), 'Hybrid rule differs unexpectedly.'
    print('Note: ML-first hybrid has the same category predictions as ML by design.')
else:
    print('No rows have valid predictions from all three models. Check the error columns.')

valid_priority=pred[pred.llm_priority.isin(PRIORITIES)]
if len(valid_priority):
    print('Qwen priority:',metrics(valid_priority.true_priority,valid_priority.llm_priority,PRIORITIES))
    print(classification_report(valid_priority.true_priority,valid_priority.llm_priority,labels=PRIORITIES,zero_division=0))
else:
    print('No valid Qwen priority predictions.')

In [ ]:
if len(common):
    for model,col in [('Traditional ML','ml_category'),('Qwen2.5 LLM','llm_category'),('Hybrid (ML-first)','hybrid_category')]:
        fig,ax=plt.subplots(figsize=(9,7))
        ConfusionMatrixDisplay.from_predictions(common.true_category,common[col],
            labels=CATEGORIES,xticks_rotation=60,ax=ax,colorbar=False)
        ax.set_title(f'{model}: category confusion matrix')
        fig.tight_layout()
        fig.savefig(OUTPUT / f'confusion_{col}.png',dpi=160,bbox_inches='tight')
        plt.show()
if len(valid_priority):
    fig,ax=plt.subplots(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(valid_priority.true_priority,valid_priority.llm_priority,
        labels=PRIORITIES,ax=ax,colorbar=False)
    ax.set_title('Qwen2.5 priority confusion matrix')
    fig.tight_layout(); fig.savefig(OUTPUT/'confusion_priority.png',dpi=160,bbox_inches='tight')
    plt.show()

## 6. Department routing and agreement analysis

Department routing is a deterministic category-to-department mapping. Its measured accuracy depends on the category decision and on whether your reference department labels match your project's department taxonomy. Agreement between ML and Qwen is **not** a substitute for ground-truth accuracy.

In [ ]:
for col in ['llm_department','hybrid_department']:
    valid=pred[col].ne('')
    if valid.any():
        print(f'{col}: {accuracy_score(pred.loc[valid,"true_department"],pred.loc[valid,col]):.2%} '
              f'over {valid.sum()} predictions')
print('ML/Qwen agreement rate:',f'{pred.models_agree.mean():.2%}')
display(pred.loc[~pred.models_agree,['complaint_id','description','true_category','ml_category','llm_category','llm_error']].head(30))

## 7. Human evaluation of generated summaries

Automatic text overlap scores are not sufficient to measure factuality. Export Qwen summaries for blinded human assessment. Have at least two reviewers independently score **factual accuracy**, **completeness**, and **clarity** from 1 (poor) to 5 (excellent), and flag hallucinations. Do not compute these scores until reviewers have filled in the CSV.

In [ ]:
review_cols=['complaint_id','description','reference_summary','llm_summary','llm_citizen_reply']
review=pred[review_cols].copy()
for name in ['reviewer1_factuality_1to5','reviewer2_factuality_1to5',
             'reviewer1_completeness_1to5','reviewer2_completeness_1to5',
             'reviewer1_clarity_1to5','reviewer2_clarity_1to5',
             'reviewer1_hallucination_yes_no','reviewer2_hallucination_yes_no']:
    review[name]=''
review_path=OUTPUT/'summary_human_review_template.csv'
review.to_csv(review_path,index=False)
print('Fill in:',review_path)
print('Important: a generated reply that promises government action without evidence is a factual/safety concern.')

## 8. Research-readiness checklist

- Replace `EXAMPLE-*` rows with independently collected, de-identified complaints.
- Confirm no overlap with ML training examples or prompts used to tune Qwen.
- Have human annotators agree on category and priority guidelines; adjudicate disagreements.
- Evaluate the entire CSV (`MAX_CASES=None`) and report both accuracy **and coverage**.
- Use the **same test cases** for ML and Qwen, and a separate development set for hybrid design.
- Report the Qwen model tag, prompts, temperature, Ollama version, machine hardware, and evaluation date.
- Review generated summaries and acknowledgements for hallucinations and unauthorized promises.
- Report confidence intervals and error analysis for any publication claims.

**Output files** are saved in `evaluation_results/`. This notebook never sends complaints to Google Gemini.